# Day 9 Data Sync and Check
Download and parse the 100-paper corpus into the `Data_V2` pipeline.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, shutil, glob, subprocess, sys, time

drive.mount("/content/drive", force_remount=True)

PROJECT_NAME = "AIGC_Fake_Detection"
GITHUB_REPO = "https://github.com/IanJ332/AIGC_Fake_Detection.git"
BRANCH = "day9-datav2-notebook-hardfix"

DRIVE_ROOT = Path("/content/drive/MyDrive/AIGC")
DATA_DIR = DRIVE_ROOT / "Data_V2"
NOTEBOOK_DIR = DRIVE_ROOT / "Notebook"
REPO_DIR = Path("/content/AIGC_Fake_Detection")
MANIFEST_PATH = "corpus/manifest.csv"

RUN_DOWNLOAD = True
RUN_PARSE = True
FORCE_REBUILD = False

USE_BACKFILL_BUILDER = True
TARGET_PARSED = 100
MAX_CANDIDATES = 300

print("BRANCH =", BRANCH)
print("DATA_DIR =", DATA_DIR)
print("MANIFEST_PATH =", MANIFEST_PATH)

In [ ]:
subdirs = [
    "pdfs", "parsed", "sections", "tables", "registry",
    "download_logs", "parse_logs", "probes", "reports",
    "checkpoints", "tei_xml"
]

if FORCE_REBUILD and DATA_DIR.exists():
    print(f"FORCE_REBUILD=True. Removing {DATA_DIR}")
    shutil.rmtree(DATA_DIR)

for name in subdirs:
    (DATA_DIR / name).mkdir(parents=True, exist_ok=True)

time.sleep(2)
print("Data_V2 folders ready:")
for name in subdirs:
    print(" -", DATA_DIR / name)

In [ ]:
!rm -rf /content/AIGC_Fake_Detection
!git clone -b day9-datav2-notebook-hardfix https://github.com/IanJ332/AIGC_Fake_Detection.git /content/AIGC_Fake_Detection
!cd /content/AIGC_Fake_Detection && git branch --show-current
!cd /content/AIGC_Fake_Detection && git log --oneline -3
!cd /content/AIGC_Fake_Detection && pip install -r requirements.txt

In [ ]:
manifest_src = REPO_DIR / MANIFEST_PATH
assert manifest_src.exists(), f"Manifest not found: {manifest_src}"

import pandas as pd
manifest_df = pd.read_csv(manifest_src)
manifest_rows = len(manifest_df)
print("Manifest rows:", manifest_rows)

shutil.copy(manifest_src, DATA_DIR / "registry" / "manifest.csv")
shutil.copy(manifest_src, DATA_DIR / "registry" / "manifest_100.csv")
print("Manifest copied to Data_V2 registry.")

In [ ]:
candidate_pool_path = REPO_DIR / "corpus" / "manifest_candidates_v2.csv"
if not candidate_pool_path.exists():
    candidate_pool_path = REPO_DIR / "artifacts" / "manifests" / "manifest_candidates.csv"

if USE_BACKFILL_BUILDER and candidate_pool_path.exists():
    print("Running Backfill Builder...")
    !cd /content/AIGC_Fake_Detection && python scripts/build_executable_corpus.py \
      --seed-manifest corpus/manifest.csv \
      --candidate-pool {candidate_pool_path.relative_to(REPO_DIR)} \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2 \
      --target-parsed {TARGET_PARSED} \
      --max-candidates {MAX_CANDIDATES} \
      --batch-size 25 \
      --delay 1.5
else:
    print("Backfill disabled or candidate pool missing. Running static download...")
    if not candidate_pool_path.exists():
        with open(REPO_DIR / "docs" / "day9_backfill_unavailable.md", "w") as f:
            f.write("# Day 9 Backfill Unavailable\nCandidate pool missing.\n")
    !cd /content/AIGC_Fake_Detection && python scripts/download_corpus.py \
      --manifest corpus/manifest.csv \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2
    
    !cd /content/AIGC_Fake_Detection && python -m src.parse.parse_pdfs \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2
    
    !cd /content/AIGC_Fake_Detection && python -m src.parse.segment_sections \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2
    
    !cd /content/AIGC_Fake_Detection && python -m src.parse.extract_table_candidates \
      --sections /content/drive/MyDrive/AIGC/Data_V2/sections/sections.jsonl \
      --out /content/drive/MyDrive/AIGC/Data_V2/tables/table_candidates.jsonl

In [ ]:
import pandas as pd, glob, os

pdf_files = sorted((DATA_DIR / "pdfs").glob("*.pdf"))
zero_byte_pdfs = [p for p in pdf_files if p.stat().st_size == 0]
registry_path = DATA_DIR / "registry" / "document_registry.csv"

print("PDF count:", len(pdf_files))
print("Zero-byte PDFs:", len(zero_byte_pdfs))
print("Registry exists:", registry_path.exists())

if registry_path.exists():
    reg = pd.read_csv(registry_path)
    print("Registry rows:", len(reg))
    if "download_status" in reg.columns:
        print(reg["download_status"].value_counts(dropna=False).to_string())

In [ ]:
import json, glob, os
from pathlib import Path

pdf_count = len(list((DATA_DIR / "pdfs").glob("*.pdf")))
parsed_json_count = len(list((DATA_DIR / "parsed").glob("*.json")))
zero_byte_count = len([p for p in (DATA_DIR / "pdfs").glob("*.pdf") if p.stat().st_size == 0])
sections_exists = (DATA_DIR / "sections" / "sections.jsonl").exists()
tables_exists = (DATA_DIR / "tables" / "table_candidates.jsonl").exists()
registry_exists = (DATA_DIR / "registry" / "document_registry.csv").exists()
parse_registry_exists = (DATA_DIR / "registry" / "parse_registry.csv").exists()

status = "BLOCKED"
if (DATA_DIR / "probes" / "executable_corpus_status.json").exists():
    with open(DATA_DIR / "probes" / "executable_corpus_status.json", "r") as f:
        st = json.load(f)
        status = st.get("status", "BLOCKED")
else:
    if parsed_json_count >= 95:
        status = "READY_FOR_NOTEBOOK_02"
    elif 70 <= parsed_json_count < 95:
        status = "CAUTION_READY_FOR_NOTEBOOK_02"

summary = {
    "branch": BRANCH,
    "data_dir": str(DATA_DIR),
    "manifest_path": MANIFEST_PATH,
    "manifest_rows": manifest_rows,
    "pdf_count": pdf_count,
    "zero_byte_pdf_count": zero_byte_count,
    "parsed_json_count": parsed_json_count,
    "sections_exists": sections_exists,
    "tables_exists": tables_exists,
    "registry_exists": registry_exists,
    "parse_registry_exists": parse_registry_exists,
    "final_status": status,
}

print(json.dumps(summary, indent=2))

with open(DATA_DIR / "probes" / "day9_data_status.json", "w") as f:
    json.dump(summary, f, indent=2)

with open(DATA_DIR / "reports" / "day9_data_sync_report.md", "w") as f:
    f.write("# Day 9 Data Sync Report\n\n")
    for k, v in summary.items():
        f.write(f"- **{k}**: {v}\n")

print("Saved:")
print(DATA_DIR / "probes" / "day9_data_status.json")
print(DATA_DIR / "reports" / "day9_data_sync_report.md")
print("FINAL STATUS:", status)